In [2]:
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers pypdf

In [3]:
!sudo apt-get update -qq
!sudo apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 85 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (12.2 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf

In [4]:
# Ahora sí, instalar Ollama dentro de la máquina virtual de Colab
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
import subprocess
import time

# Arrancar el servicio de Ollama en segundo plano
proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # dar tiempo a que el servicio levante

print("Ollama corriendo en segundo plano.")

Ollama corriendo en segundo plano.


In [6]:
# Descargar el modelo (puede tardar varios minutos, pesa varios GB)
!ollama pull llama3

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Cargar el PDF
ruta_pdf = "/content/drive/MyDrive/data/manual_calidad_arroz.pdf"
loader = PyPDFLoader(ruta_pdf)
documentos = loader.load()

print(f"Documento cargado. Total de páginas: {len(documentos)}")

/tmp/ipykernel_5866/3235496640.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Mounted at /content/drive
Documento cargado. Total de páginas: 1


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
textos = text_splitter.split_documents(documentos)

print(f"Total de fragmentos generados: {len(textos)}")

Total de fragmentos generados: 4


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Embeddings locales (multilingüe, funciona bien en español)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Crear la base de datos vectorial
db = Chroma.from_documents(textos, embeddings)

print("Base de datos vectorial creada correctamente.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Base de datos vectorial creada correctamente.


In [11]:
!pip show langchain

Name: langchain
Version: 1.3.11
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [13]:
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma langchain-ollama chromadb sentence-transformers pypdf

In [14]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Modelo local vía Ollama
llm = OllamaLLM(model="llama3")

# Retriever (busca los fragmentos más relevantes del PDF)
retriever = db.as_retriever()

# Prompt que combina el contexto recuperado con la pregunta
prompt = ChatPromptTemplate.from_template("""
Responde la pregunta basándote únicamente en el siguiente contexto.
Si no encuentras la respuesta en el contexto, dilo claramente.

Contexto:
{context}

Pregunta: {question}
""")

def formatear_documentos(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Cadena RAG usando LCEL
qa_chain = (
    {"context": retriever | formatear_documentos, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Cadena RAG lista para consultas.")

Cadena RAG lista para consultas.


In [15]:
pregunta = "¿Cuáles son los estándares de humedad aceptados para el lote de arroz?"

respuesta = qa_chain.invoke(pregunta)

print("PREGUNTA:", pregunta)
print("\nRESPUESTA:", respuesta)

# Si quieres ver también las fuentes usadas:
docs_relevantes = retriever.invoke(pregunta)
print("\n--- Fuentes utilizadas ---")
for i, doc in enumerate(docs_relevantes, 1):
    pagina = doc.metadata.get("page", "N/A")
    print(f"{i}. Página {pagina}: {doc.page_content[:150]}...")

PREGUNTA: ¿Cuáles son los estándares de humedad aceptados para el lote de arroz?

RESPUESTA: Según el contexto, los estándares de humedad aceptados para el lote de arroz paddy son del 12.0% al 13.5%. Humedades superiores al 14% rechazan automáticamente el lote por riesgo de proliferación de hongos.

--- Fuentes utilizadas ---
1. Página 0: Manual  Calidad  del  Arroz.  
1.  OBJETIVO  Establecer  los  criterios  técnicos  para  la  recepción,  inspección  y  almacenamiento  
de
 
arroz
 
...
2. Página 0: proveedor,
 
porcentaje
 
de
 
humedad
 
y
 
observaciones
 
de
 
textura.
 4.  ACCIONES  CORRECTIVAS  ●  Si  el  lote  supera  el  14%  de  humedad: ...
3. Página 0: por
 
riesgo
 
de
 
proliferación
 
de
 
hongos.
 ●  Grano  Yesoso:  Se  considera  grano  yesoso  aquel  con  aspecto  opaco  y  harinoso.  El  límit...
4. Página 0: máximo
 
tolerable
 
es
 
del
 
3%
 
por
 
lote.
 ●  Grano  Rojo/Manchado:  No  debe  exceder  el  1%  de  presencia  en  la  muestra  total.  3.  PRO...
